In [1]:
import pandas as pd
import numpy as np

# Сколько позитивов вообще попадает в кандидатов?
future = pd.read_parquet('artifacts/history_interactions.parquet')  # это history
# Нам нужен future - пересечём candidates с future targets

train_base = pd.read_parquet('artifacts/train_reranker_base.parquet')
test_base = pd.read_parquet('artifacts/test_reranker_base.parquet')
all_base = pd.concat([train_base, test_base])

# Все future interactions
interactions = pd.read_csv('data/user_games_target.csv')
interactions = interactions.sort_values(['steamid', 'rtime_last_played'])
interactions['rank_time'] = interactions.groupby('steamid').cumcount() + 1
interactions['total_games'] = interactions.groupby('steamid')['steamid'].transform('count')
interactions['is_future'] = interactions['rank_time'] > (interactions['total_games'] * 0.8)

future_all = interactions[interactions['is_future']][['steamid', 'appid', 'target']]
future_all = future_all[future_all['steamid'].isin(all_base['steamid'].unique())]

# Recall: сколько future items попало в candidates
future_in_cands = future_all.merge(all_base[['steamid', 'appid']], on=['steamid', 'appid'], how='inner')

total_future = len(future_all)
found_future = len(future_in_cands)

print(f"Total future interactions: {total_future}")
print(f"Found in candidates (top-300 ALS): {found_future}")
print(f"Recall@300: {found_future / total_future:.4f}")

# По пользователям
user_future_count = future_all.groupby('steamid').size().reset_index(name='total_future')
user_found_count = future_in_cands.groupby('steamid').size().reset_index(name='found_in_cands')
user_recall = user_future_count.merge(user_found_count, on='steamid', how='left')
user_recall['found_in_cands'] = user_recall['found_in_cands'].fillna(0)
user_recall['recall'] = user_recall['found_in_cands'] / user_recall['total_future']

print(f"\nPer-user Recall@300:")
print(f"  Mean:   {user_recall['recall'].mean():.4f}")
print(f"  Median: {user_recall['recall'].median():.4f}")
print(f"  Users with 0 recall: {(user_recall['recall'] == 0).sum()}")
print(f"  Users with recall=1: {(user_recall['recall'] == 1).sum()}")
print(f"  Distribution:")
print(user_recall['recall'].describe())

# Recall для high-target items (4-5)
future_high = future_all[future_all['target'] >= 4]
found_high = future_high.merge(all_base[['steamid', 'appid']], on=['steamid', 'appid'], how='inner')
print(f"\nRecall@300 для target>=4: {len(found_high) / len(future_high):.4f}")

# Recall для target=5
future_5 = future_all[future_all['target'] == 5]
found_5 = future_5.merge(all_base[['steamid', 'appid']], on=['steamid', 'appid'], how='inner')
print(f"Recall@300 для target=5:  {len(found_5) / len(future_5):.4f}")


Total future interactions: 87726
Found in candidates (top-300 ALS): 10868
Recall@300: 0.1239

Per-user Recall@300:
  Mean:   0.1900
  Median: 0.1579
  Users with 0 recall: 786
  Users with recall=1: 42
  Distribution:
count    4435.000000
mean        0.190034
std         0.172845
min         0.000000
25%         0.066667
50%         0.157895
75%         0.250000
max         1.000000
Name: recall, dtype: float64

Recall@300 для target>=4: 0.0835
Recall@300 для target=5:  0.0685
